# CUE 02: importing YAML and validating manifests

`cue import` turns existing YAML or JSON into CUE; `cue vet -d '#Def' schema.cue data.yaml` checks data files against a definition without converting them. A schema you write once validates every rendered manifest, whatever produced it.


In [ ]:
export HOME=/tmp
mkdir -p /source/work && cd /source/work && [ -d gitops-renderers ] || git clone -q --recurse-submodules https://github.com/cznewt/gitops-renderers.git
cd /source/work/gitops-renderers && yq 'select(.kind == "Deployment" and .metadata.name == "web")' rendered/kustomize/prod.yaml > /source/work/cue-lab/deploy.yaml && head -12 /source/work/cue-lab/deploy.yaml


In [ ]:
cd /source/work/cue-lab
cue import -p lab -f deploy.yaml && head -20 deploy.cue && rm deploy.cue


In [ ]:
cd /source/work/cue-lab
cat > k8s.cue <<'CUE'
package lab

#Deployment: {
    apiVersion: "apps/v1"
    kind:       "Deployment"
    metadata: {name: string & =~"^[a-z][a-z0-9-]*$", namespace?: string, labels?: [string]: string}
    spec: {
        replicas: int & >0 & <=20
        selector: matchLabels: [string]: string
        template: {
            metadata?: {...}
            spec: {
                containers: [...{
                    name:  string
                    image: string & !~":latest$"
                    ...
                }]
                ...
            }
        }
        ...
    }
}
CUE
cue vet -d '#Deployment' k8s.cue deploy.yaml && echo "deploy.yaml conforms to #Deployment"


In [ ]:
cd /source/work/cue-lab
yq '.spec.replicas = "three" | .spec.template.spec.containers[0].image = "nginx:latest"' deploy.yaml > broken.yaml && (cue vet -d '#Deployment' k8s.cue broken.yaml 2>&1 | head -6) || true


Two problems, two paths, before anything reaches a cluster. The same command runs over every stream in `rendered/`:


In [ ]:
cd /source/work/cue-lab
for f in /source/work/gitops-renderers/rendered/*/prod.yaml; do yq 'select(.kind == "Deployment")' $f > /tmp/d.yaml; printf '%-50s ' $f; cue vet -d '#Deployment' k8s.cue /tmp/d.yaml && echo ok; done
